In [ ]:

import json
import pennylane as qml
import pennylane.numpy as np

symbols = ["H", "H", "H"]


def h3_ground_energy(bond_length):
    
    """
    Uses VQE to calculate the ground energy of the H3+ molecule with the given bond length.
    
    Args:
        - bond_length(float): The bond length of the H3+ molecule modelled as an
        equilateral triangle.
    Returns:
        - Union[float, np.tensor, np.array]: A float-like output containing the ground 
        state of the H3+ molecule with the given bond length.
    """
    
    # Define the molecular geometry for H3+ as an equilateral triangle
    coords = np.array([
        [0.0, 0.0, 0.0],
        [bond_length, 0.0, 0.0],
        [bond_length / 2, bond_length * np.sqrt(3) / 2, 0.0]
    ])

    # H3+ has 3 H atoms, but a charge of +1, so it has 2 electrons.
    # With the sto-3g basis set, each H atom contributes one 1s orbital,
    # resulting in 3 spatial molecular orbitals in total.
    num_electrons = 2 
    num_spatial_orbitals = 3 
    # The total number of qubits (spin orbitals) is twice the number of spatial orbitals.
    num_wires = 2 * num_spatial_orbitals 

    # Build the molecular Hamiltonian for the specified active space
    H, _ = qml.qchem.molecular_hamiltonian(
        symbols,
        coords,
        charge=1,
        basis="sto-3g",
        active_electrons=num_electrons,
        active_orbitals=num_spatial_orbitals,
        mapping="jordan_wigner"
    )

    # Define the Hartree-Fock state as the initial state for VQE
    hf_state = qml.qchem.hf_state(num_electrons, num_wires)

    # Initialize the quantum device
    dev = qml.device("lightning.qubit", wires=num_wires)

    # Define the VQE circuit using the UCCSD ansatz
    @qml.qnode(dev, interface="autograd")
    def circuit(params):
        qml.BasisState(hf_state, wires=range(num_wires))
        qml.UCCSD(params, wires=range(num_wires),
                  active_electrons=num_electrons, active_orbitals=num_spatial_orbitals)
        return qml.expval(H)

    # Calculate the number of parameters required by the UCCSD ansatz
    s_excitations, d_excitations = qml.qchem.excitations(
        num_electrons, num_wires, active_orbitals=num_spatial_orbitals
    )
    
    num_params = len(s_excitations) + len(d_excitations)
    # Initialize parameters to zero
    params = np.zeros(num_params, requires_grad=True)

    # Choose an optimizer
    opt = qml.GradientDescentOptimizer(stepsize=0.2) 

    # Set optimization parameters
    max_iterations = 200 
    conv_tol = 1e-6 

    # Run the VQE optimization loop
    prev_energy = circuit(params) + 2 * conv_tol # Initialize to ensure loop runs at least once

    for n in range(max_iterations):
        params, energy = opt.step_and_cost(circuit, params)

        # Check for convergence
        if np.abs(energy - prev_energy) < conv_tol:
            break
        prev_energy = energy
    
    return float(energy)


def run(test_case_input: str) -> str:
    ins = json.loads(test_case_input)
    outs = h3_ground_energy(ins)
    return str(outs)


def check(solution_output: str, expected_output: str) -> None:
    solution_output = json.loads(solution_output)
    expected_output = json.loads(expected_output)
    assert np.allclose(solution_output,expected_output, atol = 1e-4), "Not the correct ground energy"


test_cases = [
    ('1.5', '-1.232574'),
    ('0.8', '-0.3770325')
]

if __name__ == '__main__':
    for i, (input_, expected_output) in enumerate(test_cases):
        print(f"Running test case {i} with input '{input_}'...")

        try:
            output = run(input_)

        except Exception as exc:
            print(f"Runtime Error. {exc}")

        else:
            if message := check(output, expected_output):
                print(f"Wrong Answer. Have: '{output}'. Want: '{expected_output}'.")

            else:
                print("Correct!")
